In [8]:
import os
import sys
import glob
import numpy as np
import torch
import yaml
from pathlib import Path
sys.path.append('../src')

# Try importing the necessary modules
try:
    from enformer_temp import EnformerEmbeddingsDataLoader, NumpyFilesDataset, EnformerWithEmbeddings
    print("✅ Successfully imported necessary modules")
except ImportError as e:
    print(f"❌ Error importing modules: {e}")

# Check if PyTorch is available
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

✅ Successfully imported necessary modules
PyTorch version: 2.5.1
CUDA available: False


/grid/koo/home/lemanczyk/miniconda3/envs/architecture_search_env/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [10]:
# Load config file
config_path = '../configs/test_config.yaml'
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        cfg = yaml.safe_load(f)
    print(f"✅ Config file loaded successfully")
    
    # Check important paths
    print("\nChecking data paths:")
    for path_key in ['data_path', 'train_loader', 'val_loader', 'test_loader']:
        path = cfg.get(path_key)
        if path and os.path.exists(path):
            print(f"✅ {path_key}: {path} exists")
        else:
            print(f"❌ {path_key}: {path} not found")
    
    # Check critical parameters
    print("\nCritical parameters:")
    print(f"Target layer: {cfg.get('target_layer')}")
    print(f"File indices - Train: {cfg.get('train_indices')}, Val: {cfg.get('val_indices')}, Test: {cfg.get('test_indices')}")
    print(f"File prefix: {cfg.get('file_prefix', 'test')}")
    
    # Check output directory
    outpath = cfg.get('outpath', './out/')
    if not os.path.exists(outpath):
        print(f"⚠️ Output directory {outpath} doesn't exist. Creating it.")
        os.makedirs(outpath, exist_ok=True)
    else:
        print(f"✅ Output directory {outpath} exists")
else:
    print(f"❌ Config file {config_path} not found")

✅ Config file loaded successfully

Checking data paths:
✅ data_path: /grid/koo/home/lemanczyk/SAE_Hackathon/data/enformer_data_partitioned exists
✅ train_loader: /grid/koo/home/lemanczyk/SAE_Hackathon/data/enformer_data_partitioned/human/train exists
✅ val_loader: /grid/koo/home/lemanczyk/SAE_Hackathon/data/enformer_data_partitioned/human/valid exists
✅ test_loader: /grid/koo/home/lemanczyk/SAE_Hackathon/data/enformer_data_partitioned/human/test exists

Critical parameters:
Target layer: conv_tower.5.2.to_attn_logits
File indices - Train: 1-4, Val: 5-6, Test: 7-8
File prefix: test
⚠️ Output directory ./out/ doesn't exist. Creating it.


In [11]:
# Test file loading with a very small set of indices
data_dir = cfg.get('train_loader')
test_indices = "1-3"  # Just try to load 3 files
file_prefix = cfg.get('file_prefix', 'test')

try:
    test_dataset = NumpyFilesDataset(
        data_dir,
        file_indices=test_indices,
        file_prefix=file_prefix
    )
    
    print(f"✅ Dataset initialized successfully")
    print(f"Found {len(test_dataset)} samples")
    
    if len(test_dataset.file_paths) > 0:
        print(f"Sample file paths: {test_dataset.file_paths}")
        
        # Try loading one sample
        try:
            sequence, target = test_dataset[0]
            print(f"✅ Successfully loaded a sample")
            print(f"Sequence shape: {sequence.shape}")
            print(f"Target shape: {target.shape}")
        except Exception as e:
            print(f"❌ Error loading a sample: {e}")
    else:
        print("❌ No files found with the specified pattern and indices")
except Exception as e:
    print(f"❌ Error initializing dataset: {e}")

Found 1 NumPy files
Sample file paths: ['/grid/koo/home/lemanczyk/SAE_Hackathon/data/enformer_data_partitioned/human/train/test_1.npz']
✅ Dataset initialized successfully
Found 1 samples
Sample file paths: ['/grid/koo/home/lemanczyk/SAE_Hackathon/data/enformer_data_partitioned/human/train/test_1.npz']
✅ Successfully loaded a sample
Sequence shape: torch.Size([196608, 4])
Target shape: torch.Size([896, 5313])


In [12]:
# Test if EnformerWithEmbeddings class is correctly defined and can be initialized
# This won't load the full model to save memory
try:
    # Just check if the class exists and has the right methods
    assert hasattr(EnformerWithEmbeddings, 'set_hooks'), "EnformerWithEmbeddings missing set_hooks method"
    assert hasattr(EnformerWithEmbeddings, 'get_embeddings'), "EnformerWithEmbeddings missing get_embeddings method"
    print("✅ EnformerWithEmbeddings class is properly defined with required methods")
except Exception as e:
    print(f"❌ Error with EnformerWithEmbeddings class: {e}")

✅ EnformerWithEmbeddings class is properly defined with required methods


In [13]:
# Try loading a single NPZ file to check its content structure
try:
    if len(test_dataset.file_paths) > 0:
        sample_file = test_dataset.file_paths[0]
        print(f"Examining file: {sample_file}")
        
        data = np.load(sample_file, allow_pickle=True)
        
        if isinstance(data, np.ndarray):
            print("File contains a NumPy array")
            print(f"Array shape: {data.shape}")
        else:
            print("File contains a dictionary-like structure with keys:")
            for key in data.keys():
                print(f"  - {key}: {data[key].shape if hasattr(data[key], 'shape') else type(data[key])}")
    else:
        print("❌ No files available to examine")
except Exception as e:
    print(f"❌ Error examining NPZ file: {e}")

Examining file: /grid/koo/home/lemanczyk/SAE_Hackathon/data/enformer_data_partitioned/human/train/test_1.npz
File contains a dictionary-like structure with keys:
  - sequence: (1, 196608, 4)
  - target: (1, 896, 5313)


In [14]:
# Check that the act_size in config matches the expected dimension from the target layer
act_size = cfg.get('act_size')
print(f"Config act_size: {act_size}")

# We can't easily check the actual layer dimensions without loading the model
# But we can at least verify that a value is provided and it's reasonable
if act_size is not None:
    if 100 <= act_size <= 10000:  # Reasonable range for embedding dimensions
        print(f"✅ act_size value {act_size} seems reasonable")
    else:
        print(f"⚠️ act_size value {act_size} may be unusual - very small or large")
else:
    print("❌ act_size is not defined in the config")
    
# Check SAE model parameters
dict_size = cfg.get('dict_size')
if dict_size is not None:
    ratio = dict_size / act_size if act_size else 0
    print(f"Dictionary size: {dict_size}")
    print(f"Overcomplete ratio (dict_size/act_size): {ratio:.2f}")
    if ratio > 1:
        print(f"✅ Dictionary is overcomplete (ratio > 1)")
    else:
        print(f"⚠️ Dictionary is not overcomplete (ratio <= 1)")

Config act_size: 768
✅ act_size value 768 seems reasonable
Dictionary size: 12288
Overcomplete ratio (dict_size/act_size): 16.00
✅ Dictionary is overcomplete (ratio > 1)


In [15]:
# Test the EnformerEmbeddingsDataLoader without actually loading the model
# We'll create a minimal version just to test the file loading
from torch.utils.data import DataLoader

# Create a small dataloader with just 2 samples and 1 worker
small_dataset = NumpyFilesDataset(
    data_dir,
    file_indices="1-2",
    file_prefix=file_prefix
)

try:
    small_dataloader = DataLoader(
        small_dataset,
        batch_size=1,  # Minimal batch size
        shuffle=False,
        num_workers=0  # No parallel loading
    )
    
    print(f"✅ DataLoader created successfully with {len(small_dataloader)} batches")
    
    # Try iterating through one batch without loading the model
    for sequences, targets in small_dataloader:
        print(f"Sample batch shapes - Sequences: {sequences.shape}, Targets: {targets.shape}")
        break
        
except Exception as e:
    print(f"❌ Error with DataLoader: {e}")

Found 1 NumPy files
Sample file paths: ['/grid/koo/home/lemanczyk/SAE_Hackathon/data/enformer_data_partitioned/human/train/test_1.npz']
✅ DataLoader created successfully with 1 batches
Sample batch shapes - Sequences: torch.Size([1, 196608, 4]), Targets: torch.Size([1, 896, 5313])
